In [6]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist.data / 255.0
y = mnist.target.astype(int)
def one_hot(y):
    oh = np.zeros((y.size, 10))
    oh[np.arange(y.size), y] = 1
    return oh
y_encoded = one_hot(y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)
input_size = 784
hidden_size = 128
output_size = 10
def he_init(n_in, n_out):
    return np.random.randn(n_in, n_out) * np.sqrt(2. / n_in)
W1 = he_init(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))
W2 = he_init(hidden_size, output_size)
b2 = np.zeros((1, output_size)) 
def relu(x):
    return np.maximum(0, x)
def relu_derivative(x):
    return (x > 0).astype(float)
def softmax(x):
    exp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)
lr = 0.1        
epochs = 10
batch_size = 64  
m_train = X_train.shape[0]
for epoch in range(epochs):
    indices = np.random.permutation(m_train)
    X_train_sh = X_train[indices]
    y_train_sh = y_train[indices]
    for i in range(0, m_train, batch_size):
        X_batch = X_train_sh[i : i + batch_size]
        y_batch = y_train_sh[i : i + batch_size]
        curr_m = X_batch.shape[0]
        z1 = np.dot(X_batch, W1) + b1
        a1 = relu(z1)
        z2 = np.dot(a1, W2) + b2
        a2 = softmax(z2)
        dz2 = a2 - y_batch
        dW2 = np.dot(a1.T, dz2) / curr_m
        db2 = np.sum(dz2, axis=0, keepdims=True) / curr_m
        dz1 = np.dot(dz2, W2.T) * relu_derivative(z1)
        dW1 = np.dot(X_batch.T, dz1) / curr_m
        db1 = np.sum(dz1, axis=0, keepdims=True) / curr_m
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2
    full_z1 = np.dot(X_train, W1) + b1
    full_a1 = relu(full_z1)
    full_z2 = np.dot(full_a1, W2) + b2
    full_a2 = softmax(full_z2)
    preds = np.argmax(full_a2, axis=1)
    true = np.argmax(y_train, axis=1)
    acc = np.mean(preds == true)
    print(f"Epoch {epoch+1}: Training Accuracy = {acc:.4f}")
z1_test = np.dot(X_test, W1) + b1
a1_test = relu(z1_test)
z2_test = np.dot(a1_test, W2) + b2
a2_test = softmax(z2_test)
test_preds = np.argmax(a2_test, axis=1)
test_true = np.argmax(y_test, axis=1)
test_accuracy = np.mean(test_preds == test_true)
print(f"\nFinal Test Accuracy: {test_accuracy:.4f}")


Epoch 1: Training Accuracy = 0.9333
Epoch 2: Training Accuracy = 0.9537
Epoch 3: Training Accuracy = 0.9617
Epoch 4: Training Accuracy = 0.9702
Epoch 5: Training Accuracy = 0.9727
Epoch 6: Training Accuracy = 0.9788
Epoch 7: Training Accuracy = 0.9816
Epoch 8: Training Accuracy = 0.9840
Epoch 9: Training Accuracy = 0.9846
Epoch 10: Training Accuracy = 0.9845

Final Test Accuracy: 0.9711


In [7]:
 pip install torch torchvision matplotlib

Note: you may need to restart the kernel to use updated packages.


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

100%|█████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:02<00:00, 3.75MB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 103kB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:01<00:00, 929kB/s]
100%|█████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<?, ?B/s]


In [9]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_layers, output_size, activation):
        super(MLP, self).__init__()
        
        layers = []
        prev_size = input_size
        for h in hidden_layers:
            layers.append(nn.Linear(prev_size, h))
            
            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'sigmoid':
                layers.append(nn.Sigmoid())
            elif activation == 'tanh':
                layers.append(nn.Tanh())
                
            prev_size = h
        layers.append(nn.Linear(prev_size, output_size))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.view(x.size(0), -1) 
        return self.model(x)

In [10]:
def train(model, train_loader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in range(epochs):
        total_loss = 0
        
        for images, labels in train_loader:
            optimizer.zero_grad()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [11]:
def test(model, test_loader):
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    return accuracy

In [12]:
configs = [
    ([128], 'relu'),       
    ([256, 128], 'relu'),     
    ([512, 256, 128], 'relu') 
]
for hidden, act in configs:
    print(f"\nModel: {hidden}, Activation: {act}")
    
    model = MLP(784, hidden, 10, act)
    train(model, train_loader)
    
    acc = test(model, test_loader)
    print(f"Accuracy: {acc:.2f}%")


Model: [128], Activation: relu
Epoch 1, Loss: 364.2573
Epoch 2, Loss: 192.1961
Epoch 3, Loss: 138.1549
Epoch 4, Loss: 108.9450
Epoch 5, Loss: 93.9683
Accuracy: 96.72%

Model: [256, 128], Activation: relu
Epoch 1, Loss: 319.0740
Epoch 2, Loss: 141.6522
Epoch 3, Loss: 102.2864
Epoch 4, Loss: 85.0960
Epoch 5, Loss: 70.5941
Accuracy: 96.58%

Model: [512, 256, 128], Activation: relu
Epoch 1, Loss: 305.3900
Epoch 2, Loss: 134.0791
Epoch 3, Loss: 102.7875
Epoch 4, Loss: 82.1512
Epoch 5, Loss: 70.6747
Accuracy: 97.42%


In [13]:
activations = ['relu', 'sigmoid', 'tanh']

for act in activations:
    print(f"\nActivation: {act}")
    
    model = MLP(784, [256, 128], 10, act)
    train(model, train_loader)
    
    acc = test(model, test_loader)
    print(f"Accuracy: {acc:.2f}%")


Activation: relu
Epoch 1, Loss: 319.7028
Epoch 2, Loss: 145.5688
Epoch 3, Loss: 105.7162
Epoch 4, Loss: 86.9392
Epoch 5, Loss: 71.4474
Accuracy: 96.79%

Activation: sigmoid
Epoch 1, Loss: 457.5693
Epoch 2, Loss: 162.8791
Epoch 3, Loss: 113.4460
Epoch 4, Loss: 88.7982
Epoch 5, Loss: 72.4859
Accuracy: 96.72%

Activation: tanh
Epoch 1, Loss: 292.8641
Epoch 2, Loss: 141.9572
Epoch 3, Loss: 114.5904
Epoch 4, Loss: 99.7103
Epoch 5, Loss: 97.2877
Accuracy: 96.09%
